In [59]:
    # Standard library imports
import os
import random
import gc
import copy

# Third-party library import
import numpy as np
import pandas as pd

# PyTorch and related libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader

# einops library for tensor operations
from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange, Reduce
# Custom TINTO library imports
from TINTOlib.tinto import TINTO
from TINTOlib.supertml import SuperTML
from TINTOlib.igtd import IGTD
from TINTOlib.refined import REFINED
from TINTOlib.barGraph import BarGraph
from TINTOlib.distanceMatrix import DistanceMatrix
from TINTOlib.combination import Combination
from TINTOlib.featureWrap import FeatureWrap
from TINTOlib.bie import BIE

In [60]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning) 

import torch

# Get CUDA version
cuda_version = torch.version.cuda
print(f"CUDA Version: {cuda_version}")

# Get cuDNN version
cudnn_version = torch.backends.cudnn.version()
print(f"cuDNN Version: {cudnn_version}")

# Get PyTorch version
pytorch_version = torch.__version__
print(f"PyTorch Version: {pytorch_version}")

# Check if CUDA is available
if torch.cuda.is_available():
    print("CUDA is available. PyTorch can use GPU.")
    
    # Get the name of the current GPU
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")
    
    # Create a random tensor and move it to GPU to verify
    x = torch.rand(5, 3)
    print(f"Is this tensor on GPU? {x.cuda().is_cuda}")
else:
    print("CUDA is not available. PyTorch will use CPU.")

# Additional check: is CUDA initialized?
print(f"Is CUDA initialized? {torch.cuda.is_initialized()}")

# Number of available GPUs
print(f"Number of available GPUs: {torch.cuda.device_count()}")

# Current device index
print(f"Current device index: {torch.cuda.current_device()}")


## DATASET

In [61]:
SEED = 64
# SET RANDOM SEED FOR REPRODUCIBILITY
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [62]:
# Create variable to store dataset name
dataset_name = 'ULA_lab_LoS_8'
results_path = f'./logs/Regression/{dataset_name}/ViT_Regression'

In [63]:
df = pd.read_csv(f"./ultra_dense/datasets/{dataset_name}.csv")

In [64]:
# Drop the second-to-last column if MIMO
#df = df.drop(df.columns[-2], axis=1)

In [65]:
df.shape

In [66]:
df.head()

In [67]:
# Get the last two columns
last_two_cols = df.iloc[:, -2:]

# Compute min and max for each of them
min_values = last_two_cols.min()
max_values = last_two_cols.max()

print("Minimum values:")
print(min_values)

print("\nMaximum values:")
print(max_values)


## LOAD AND PREPROCESS

In [68]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
import os
import cv2

# Function to load and preprocess data
def load_and_preprocess_data(images_folder, image_model, problem_type, batch_size=32):
    
    X_train, X_val = train_test_split(df, test_size=0.15, random_state=SEED)
    X_val, X_test = train_test_split(X_val, test_size=1/3, random_state=SEED)
    X_train = X_train.reset_index(drop=True)
    X_val = X_val.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    
    ### X_train
    # Generate the images if the folder does not exist
    if not os.path.exists(f'{images_folder}/train'):
        #Generate thet images
        image_model.fit_transform(X_train, f'{images_folder}/train')
        image_model.saveHyperparameters(f'{images_folder}'+"/model.pkl")
    else:
        print("The images are already generated")

    img_paths = os.path.join(f'{images_folder}/train',problem_type+".csv")

    print(img_paths)
    
    imgs = pd.read_csv(img_paths)
    
    # Update image paths
    imgs["images"] = images_folder + "/train/" + imgs["images"]

    # Combine datasets
    combined_dataset = pd.concat([imgs, X_train], axis=1)

    # Split data
    X_train = combined_dataset.drop(df.columns[-2:], axis=1).drop("values", axis=1)
    y_train = combined_dataset.iloc[:, -2:]
        
    ### X_val
    # Generate the images if the folder does not exist
    if not os.path.exists(f'{images_folder}/val'):
        #Generate thet images
        image_model.transform(X_val, f'{images_folder}/val')
    else:
        print("The images are already generated")

    img_paths = os.path.join(f'{images_folder}/val',problem_type+".csv")

    print(img_paths)
    
    imgs = pd.read_csv(img_paths)

    # Update image paths
    imgs["images"] = images_folder + "/val/" + imgs["images"]

    # Combine datasets
    combined_dataset = pd.concat([imgs, X_val], axis=1)

    # Split data
    X_val = combined_dataset.drop(df.columns[-2:], axis=1).drop("values", axis=1)
    y_val = combined_dataset.iloc[:, -2:]
    
    ### X_test
    # Generate the images if the folder does not exist
    if not os.path.exists(f'{images_folder}/test'):
        #Generate thet images
        image_model.transform(X_test, f'{images_folder}/test')
    else:
        print("The images are already generated")

    img_paths = os.path.join(f'{images_folder}/test',problem_type+".csv")

    print(img_paths)
    
    imgs = pd.read_csv(img_paths)

    # Update image paths
    imgs["images"] = images_folder + "/test/" + imgs["images"]

    # Combine datasets
    combined_dataset = pd.concat([imgs, X_test], axis=1)

    # Split data
    X_test = combined_dataset.drop(df.columns[-2:], axis=1).drop("values", axis=1)
    y_test = combined_dataset.iloc[:, -2:]
    
    # Numerical data
    X_train_num = X_train.drop("images", axis=1)
    X_val_num = X_val.drop("images", axis=1)
    X_test_num = X_test.drop("images", axis=1)

    # Image data
    X_train_img = np.array([cv2.imread(img) for img in X_train["images"]])
    X_val_img = np.array([cv2.imread(img) for img in X_val["images"]])
    X_test_img = np.array([cv2.imread(img) for img in X_test["images"]])

    ## Create a MinMaxScaler object
    scaler = MinMaxScaler()
#
    ## Scale numerical data
    X_train_num = pd.DataFrame(scaler.fit_transform(X_train_num), columns=X_train_num.columns)
    X_val_num = pd.DataFrame(scaler.transform(X_val_num), columns=X_val_num.columns)
    X_test_num = pd.DataFrame(scaler.transform(X_test_num), columns=X_test_num.columns)

    attributes = len(X_train_num.columns)
    height, width, channels = X_train_img[0].shape
    imgs_shape = (channels, height, width)

    print("Images shape: ", imgs_shape)
    print("Attributes: ", attributes)
    # Convert data to PyTorch tensors
    X_train_num_tensor = torch.as_tensor(X_train_num.values, dtype=torch.float32)
    X_val_num_tensor = torch.as_tensor(X_val_num.values, dtype=torch.float32)
    X_test_num_tensor = torch.as_tensor(X_test_num.values, dtype=torch.float32)
    X_train_img_tensor = torch.as_tensor(X_train_img, dtype=torch.float32).permute(0, 3, 1, 2)
    X_val_img_tensor = torch.as_tensor(X_val_img, dtype=torch.float32).permute(0, 3, 1, 2)
    X_test_img_tensor = torch.as_tensor(X_test_img, dtype=torch.float32).permute(0, 3, 1, 2)
    y_train_tensor = torch.as_tensor(y_train.values, dtype=torch.float32).reshape(-1, 2)
    y_val_tensor = torch.as_tensor(y_val.values, dtype=torch.float32).reshape(-1, 2)
    y_test_tensor = torch.as_tensor(y_test.values, dtype=torch.float32).reshape(-1, 2)

    # Normalize to [0, 1]
    #X_train_img_tensor = X_train_img_tensor / 255.0
    #X_val_img_tensor = X_val_img_tensor / 255.0
    #X_test_img_tensor = X_test_img_tensor / 255.0

    # Create DataLoaders
    train_dataset = TensorDataset(X_train_img_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_img_tensor, y_val_tensor)
    test_dataset = TensorDataset(X_test_img_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    return train_loader, val_loader, test_loader, attributes, imgs_shape 

In [69]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
import os
import cv2

# Function to load and preprocess data
def load_and_preprocess_data_test(images_folder, image_model, problem_type, batch_size=32):
    
    X_train, X_val = train_test_split(df, test_size=0.15, random_state=SEED)
    X_val, X_test = train_test_split(X_val, test_size=1/3, random_state=SEED)
    X_train = X_train.reset_index(drop=True)
    X_val = X_val.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    
    ### X_train
    # Generate the images if the folder does not exist
    if not os.path.exists(f'{images_folder}/train'):
        #Generate thet images
        image_model.fit_transform(X_train, f'{images_folder}/train')
        image_model.saveHyperparameters(f'{images_folder}'+"/model.pkl")
    else:
        print("The images are already generated")

    img_paths = os.path.join(f'{images_folder}/train',problem_type+".csv")

    print(img_paths)
    
    imgs = pd.read_csv(img_paths)
    
    # Update image paths
    imgs["images"] = images_folder + "/train/" + imgs["images"]

    # Combine datasets
    combined_dataset = pd.concat([imgs, X_train], axis=1)

    # Split data
    X_train = combined_dataset.drop(df.columns[-2:], axis=1).drop("values", axis=1)
    y_train = combined_dataset.iloc[:, -2:]
        
    ### X_val
    # Generate the images if the folder does not exist
    if not os.path.exists(f'{images_folder}/val'):
        #Generate thet images
        image_model.transform(X_val, f'{images_folder}/val')
    else:
        print("The images are already generated")

    img_paths = os.path.join(f'{images_folder}/val',problem_type+".csv")

    print(img_paths)
    
    imgs = pd.read_csv(img_paths)

    # Update image paths
    imgs["images"] = images_folder + "/val/" + imgs["images"]

    # Combine datasets
    combined_dataset = pd.concat([imgs, X_val], axis=1)

    # Split data
    X_val = combined_dataset.drop(df.columns[-2:], axis=1).drop("values", axis=1)
    y_val = combined_dataset.iloc[:, -2:]
    
    ### X_test
    # Generate the images if the folder does not exist
    if not os.path.exists(f'{images_folder}/test'):
        #Generate thet images
        image_model.transform(X_test, f'{images_folder}/test')
    else:
        print("The images are already generated")

    img_paths = os.path.join(f'{images_folder}/test',problem_type+".csv")

    print(img_paths)
    
    imgs = pd.read_csv(img_paths)

    # Update image paths
    imgs["images"] = images_folder + "/test/" + imgs["images"]

    # Combine datasets
    combined_dataset = pd.concat([imgs, X_test], axis=1)

    # Split data
    X_test = combined_dataset.drop(df.columns[-2:], axis=1).drop("values", axis=1)
    y_test = combined_dataset.iloc[:, -2:]
    
    # Numerical data
    X_train_num = X_train.drop("images", axis=1)
    X_val_num = X_val.drop("images", axis=1)
    X_test_num = X_test.drop("images", axis=1)

    # Image data
    X_train_img = None
    X_val_img = None
    X_test_img = np.array([cv2.imread(img) for img in X_test["images"]])

    ## Create a MinMaxScaler object
    scaler = MinMaxScaler()
#
    ## Scale numerical data
    X_train_num = pd.DataFrame(scaler.fit_transform(X_train_num), columns=X_train_num.columns)
    X_val_num = pd.DataFrame(scaler.transform(X_val_num), columns=X_val_num.columns)
    X_test_num = pd.DataFrame(scaler.transform(X_test_num), columns=X_test_num.columns)

    attributes = len(X_test_num.columns)
    height, width, channels = X_test_img[0].shape
    imgs_shape = (channels, height, width)

    print("Images shape: ", imgs_shape)
    print("Attributes: ", attributes)
    # Convert data to PyTorch tensors
    X_test_num_tensor = torch.as_tensor(X_test_num.values, dtype=torch.float32)
    X_test_img_tensor = torch.as_tensor(X_test_img, dtype=torch.float32).permute(0, 3, 1, 2)
    y_test_tensor = torch.as_tensor(y_test.values, dtype=torch.float32).reshape(-1, 2)

    # Normalize to [0, 1]
    #X_train_img_tensor = X_train_img_tensor / 255.0
    #X_val_img_tensor = X_val_img_tensor / 255.0
    #X_test_img_tensor = X_test_img_tensor / 255.0

    # Create DataLoaders
    train_dataset = None
    val_dataset = None
    test_dataset = TensorDataset(X_test_img_tensor, y_test_tensor)

    train_loader = None
    val_loader = None
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    return train_loader, val_loader, test_loader, attributes, imgs_shape 

## MODEL ARCHITECTURES

In [70]:
from vit_pytorch.vit_attn import ViT

In [71]:
def find_divisors(n):
    divisors = []
    for i in range(1, int(n**0.5) + 1):
        if n % i == 0:
            divisors.append(i)
            if i != n // i:  # Check to include both divisors if they are not the same
                divisors.append(n // i)
    divisors.sort()
    return divisors

In [72]:
class Model1(nn.Module):
    def __init__(self, attributes, imgs_shape, patch_size):
        super(Model1, self).__init__()
        
        # ViT branch
        self.vit = ViT(
            image_size = imgs_shape,
            patch_size = patch_size,
            dim = 32,
            depth = 2,
            heads = 4,
            mlp_dim = 64,
            dropout = 0.1,
            emb_dropout = 0.1
        )
        
        # MLP branch
        self.mlp = nn.Sequential(
            nn.Linear(attributes, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
        )

        
        # Final MLP
        self.final_mlp = nn.Sequential(
            nn.Linear(32+8, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
        )

    def forward(self, mlp_input, vit_input):
        vit_output = self.vit(vit_input)
        mlp_output = self.mlp(mlp_input)
        concat_output = torch.cat((mlp_output, vit_output), dim=1)
        return self.final_mlp(concat_output)

In [73]:
class Model2(nn.Module):
    def __init__(self, attributes, imgs_shape, patch_size):
        super(Model2, self).__init__()
        
        # ViT branch
        self.vit = ViT(
            image_size = imgs_shape,
            patch_size = patch_size,
            dim = 512,  # Increased dimensionality
            depth = 4,  # More layers
            heads = 8,  # More attention heads
            mlp_dim = 512,
            dropout = 0.0,
            emb_dropout = 0.0
        )
        
        # MLP branch
        self.mlp = nn.Sequential(
            nn.Linear(attributes, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
        )
        
        # Final MLP
        self.final_mlp = nn.Sequential(
            nn.Linear(512+16, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
        )

    def forward(self, mlp_input, vit_input):
        vit_output = self.vit(vit_input)
        mlp_output = self.mlp(mlp_input)
        concat_output = torch.cat((mlp_output, vit_output), dim=1)
        return self.final_mlp(concat_output)

In [74]:
class Model4(nn.Module):
    def __init__(self, imgs_shape, patch_size):
        super(Model4, self).__init__()
        
        # Enhanced ViT branch with increased depth and heads
        self.vit = ViT(
            image_size = imgs_shape,
            patch_size = patch_size,
            dim = 512,  # Increased dimensionality
            depth = 4,  # More layers
            heads = 8,  # More attention heads
            mlp_dim = 512,
            dropout = 0.0,
            emb_dropout = 0.0
        )
        
        # More complex MLP
        self.mlp = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, vit_input, return_attn=False):
        if return_attn:
            vit_output, attn_maps = self.vit(vit_input, return_attn=return_attn)
        else:
            vit_output = self.vit(vit_input)
            
        output = self.mlp(vit_output)

        return (output, attn_maps) if return_attn else output 

## COMPILE AND FIT

In [75]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch.optim.lr_scheduler import OneCycleLR
import matplotlib.pyplot as plt
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os

def compile_and_fit(model, train_loader, val_loader, test_loader, dataset_name, model_name, batch_size=32, epochs=100, min_lr=1e-3, max_lr=1, device='cuda', weight_decay=1e-2):
    model = model.to(device)
    loss_fn = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=min_lr, weight_decay=weight_decay)
    
    total_steps = epochs * len(train_loader)
    scheduler = OneCycleLR(optimizer, max_lr=max_lr, div_factor=max_lr/min_lr, total_steps=total_steps, pct_start=0.3, final_div_factor=1)
    
    
    best_val_loss = float('inf')
    best_model = None
    best_epoch = 0
    #early_stopping_counter = 0
    #early_stopping_patience = 20
    #warm_up_epochs = epochs*0.3

    history = {'train_loss': [], 'val_loss': [], 'train_mse': [], 'val_mse': [], 'train_rmse': [], 'val_rmse': [], 'learning_rate': [], 'epoch_time': []}

    start_time = time.time()
    
    for epoch in range(epochs):
        epoch_start_time = time.time()

        model.train()
        train_loss = 0.0
        train_predictions = []
        train_targets = []
        for num_data, img_data, targets in train_loader:
            num_data, img_data, targets = num_data.to(device, non_blocking=True), img_data.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            outputs = model(num_data, img_data)
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            train_predictions.extend(outputs.cpu().detach().numpy())
            train_targets.extend(targets.cpu().numpy())

        model.eval()
        val_loss = 0.0
        val_predictions = []
        val_targets = []
        with torch.no_grad():
            for num_data, img_data, targets in val_loader:
                num_data, img_data, targets = num_data.to(device, non_blocking=True), img_data.to(device, non_blocking=True), targets.to(device, non_blocking=True)
                outputs = model(num_data, img_data)
                loss = loss_fn(outputs, targets)
                
                val_loss += loss.item()
                val_predictions.extend(outputs.cpu().numpy())
                val_targets.extend(targets.cpu().numpy())

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        
        # Get the current learning rate
        current_lr = scheduler.get_last_lr()
        
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            #early_stopping_counter = 0
        #else:
            #if epoch > warm_up_epochs:
                #early_stopping_counter += 1
                #if early_stopping_counter >= early_stopping_patience:
                    #print(f"Early stopping triggered at epoch {epoch+1}")
                    #break

        train_mse = mean_squared_error(train_targets, train_predictions)
        train_rmse = np.sqrt(train_mse)
        val_mse = mean_squared_error(val_targets, val_predictions)
        val_rmse = np.sqrt(val_mse)
        train_r2 = r2_score(train_targets, train_predictions)
        val_r2 = r2_score(val_targets, val_predictions)

        epoch_time = time.time() - epoch_start_time

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_mse'].append(train_mse)
        history['val_mse'].append(val_mse)
        history['train_rmse'].append(train_rmse)
        history['val_rmse'].append(val_rmse)
        history['learning_rate'].append(current_lr)
        history['epoch_time'].append(epoch_time)

    total_time = time.time() - start_time
    model.load_state_dict(best_model)

    # Calculate and save metrics
    train_metrics = calculate_metrics(model, train_loader, device)
    val_metrics = calculate_metrics(model, val_loader, device)
    test_metrics = calculate_metrics(model, test_loader, device)

    metrics = {
        'train_loss': train_metrics['loss'],
        'train_mse': train_metrics['mse'],
        'train_mae': train_metrics['mae'],
        'train_rmse': train_metrics['rmse'],
        'train_r2': train_metrics['r2'],
        'val_loss': val_metrics['loss'],
        'val_mse': val_metrics['mse'],
        'val_mae': val_metrics['mae'],
        'val_rmse': val_metrics['rmse'],
        'val_r2': val_metrics['r2'],
        'test_loss': test_metrics['loss'],
        'test_mse': test_metrics['mse'],
        'test_mae': test_metrics['mae'],
        'test_rmse': test_metrics['rmse'],
        'test_r2': test_metrics['r2'],
        'min_lr': min_lr,
        'max_lr': max_lr,
        'total_time': total_time,
        'average_epoch_time': sum(history['epoch_time']) / len(history['epoch_time'])
    }
    
    print(f"\nTraining completed in {total_time:.2f} seconds")
    print(f"Best model found at epoch {best_epoch}/{epochs}")
    print(f"Best Train Loss: {history['train_loss'][best_epoch-1]:.4f}, Best Val Loss: {history['val_loss'][best_epoch-1]:.4f}")
    print(f"Best Train MSE: {history['train_mse'][best_epoch-1]:.4f}, Best Val MSE: {history['val_mse'][best_epoch-1]:.4f}")
    print(f"Best Train RMSE: {history['train_rmse'][best_epoch-1]:.4f}, Best Val RMSE: {history['val_rmse'][best_epoch-1]:.4f}")

    # Save figures for this fold
    os.makedirs(f"models/Regression/{dataset_name}/ViT+MLP/{model_name}", exist_ok=True)
    plot_metric(history['train_loss'], history['val_loss'], 'Loss', dataset_name, model_name)
    plot_metric(history['train_mse'], history['val_mse'], 'MSE', dataset_name, model_name)
    plot_metric(history['train_rmse'], history['val_rmse'], 'RMSE', dataset_name, model_name)
    plot_learning_rate(history['learning_rate'], dataset_name, model_name)

    # Save metrics to a file
    os.makedirs(f'logs/Regression/{dataset_name}/ViT+MLP/{model_name}', exist_ok=True)
    with open(f'logs/Regression/{dataset_name}/ViT+MLP/{model_name}/metrics.txt', 'w') as f:
        for key, value in metrics.items():
            f.write(f'{key}: {value}\n')
            
    # Save best model
    model_save_path = f"models/Regression/{dataset_name}/ViT+MLP/{model_name}/best_model.pth"
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    torch.save(best_model, model_save_path)
    print(f"Best model saved to {model_save_path}")
            
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return metrics

def calculate_metrics(model, data_loader, device):
    model.eval()
    loss_fn = nn.MSELoss()
    total_loss = 0
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for num_data, img_data, targets in data_loader:
            num_data, img_data, targets = num_data.to(device, non_blocking=True), img_data.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            outputs = model(num_data, img_data)
            loss = loss_fn(outputs, targets)
            total_loss += loss.item()
            all_targets.append(targets.cpu().numpy())
            all_predictions.append(outputs.cpu().numpy())

    all_targets = np.concatenate(all_targets)
    all_predictions = np.concatenate(all_predictions)

    mse = mean_squared_error(all_targets, all_predictions, multioutput='raw_values')
    mae = mean_absolute_error(all_targets, all_predictions, multioutput='raw_values')
    rmse = np.sqrt(mse)
    r2 = r2_score(all_targets, all_predictions, multioutput='raw_values')

    metrics = {
        'loss': total_loss / len(data_loader.dataset),
        'mse': mse,
        'mae': mae,
        'rmse': rmse,
        'r2': r2
    }
    return metrics    

def plot_metric(train_metric, val_metric, metric_name, dataset_name, model_name):
    plt.figure()
    plt.plot(train_metric, label=f'Train {metric_name}')
    plt.plot(val_metric, label=f'Validation {metric_name}')
    plt.xlabel('Epoch')
    plt.ylabel(metric_name)
    plt.legend()
    plt.title(f'{metric_name} vs. Epoch')
    plt.savefig(f"models/Regression/{dataset_name}/ViT+MLP/{model_name}/{metric_name.lower()}_plot.png")
    plt.close()

def plot_learning_rate(learning_rates, dataset_name, model_name):
    plt.figure()
    plt.plot(learning_rates)
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.title('Learning Rate vs. Epoch')
    plt.savefig(f"models/Regression/{dataset_name}/ViT+MLP/{model_name}/learning_rate_plot.png")
    plt.close()

In [76]:
def safe_compile_and_fit(model, train_loader, val_loader, test_loader, dataset_name, model_name, batch_size=64, epochs=100, min_lr=1e-3, max_lr=1 , device='cuda', weight_decay=1e-2):
    try:
        if model is None:
            print(f"Model {model_name} is None")
            return None
        else:
            # Compile and fit the model
            metrics = compile_and_fit(model, train_loader, val_loader, test_loader, dataset_name, model_name, epochs=epochs, min_lr=min_lr, max_lr=max_lr, device=device, weight_decay=weight_decay)
            return metrics
    except Exception as e:
        print(f"Failed to compile and fit {model_name}: {str(e)}")
        return None
    finally:
        # Clear CUDA cache and force garbage collection
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

In [77]:
import traceback

def try_create_model(model_class, patch_size, imgs_shape):
    try:
        model = model_class(imgs_shape[1:], patch_size)
        
        # Test the model with a sample input
        sample_input = torch.randn(4, *imgs_shape)
        output = model(sample_input)
        
        print(f"Successfully created and tested {model_class.__name__}")
        
        return model
    except Exception as e:
        print(f"Error creating or testing {model_class.__name__}: {str(e)}")
        traceback.print_exc()
        return None

## EXPERIMENTS

In [78]:
#Select the model and the parameters
problem_type = "regression"
image_model = TINTO(problem= problem_type, blur=True, random_seed=SEED)
#image_model = REFINED(problem= problem_type,hcIterations=5)
#image_model = IGTD(problem= problem_type)
#image_model = BarGraph(problem= problem_type)
#image_model = DistanceMatrix(problem= problem_type)
#image_model = Combination(problem= problem_type)
#image_model = SuperTML(problem= problem_type)

#Define the dataset path and the folder where the images will be saved
images_folder = f"./HyNNImages/Regression/{dataset_name}/images_{dataset_name}_TINTO"

In [79]:
def calculate_iterations_per_epoch(dataset_size, batch_size):
    iterations = dataset_size // batch_size
    if dataset_size % batch_size != 0:
        iterations += 1
    return iterations

In [80]:
batch_size = 128

In [81]:
num_epochs = calculate_iterations_per_epoch(df.shape[0], batch_size)

In [82]:
num_epochs

### EXPERIMENT 1: TINTO

In [ ]:
#Select the model and the parameters
problem_type = "regression"
image_model = TINTO(problem= problem_type, blur=True, option="maximum", random_seed=SEED, pixels=30)
name = f"TINTO_blur_maximum"

#Define the dataset path and the folder where the images will be saved
images_folder = f"./HyNNImages/Regression/{dataset_name}/images_{dataset_name}_{name}"

In [ ]:
train_loader, val_loader, test_loader, attributes, imgs_shape  = load_and_preprocess_data(images_folder, image_model, problem_type, batch_size=batch_size)

### EXPERIMENT 2: IGTD

In [ ]:
# Get the shape of the dataframe
num_columns = df.shape[1]

# Calculate number of columns - 1
columns_minus_one = num_columns - 1

# Calculate number of columns - 2 if multi objective.
columns_minus_one = num_columns - 2

# Calculate the square root for image size
import math
image_size = math.ceil(math.sqrt(columns_minus_one))
print(image_size)

In [ ]:
#Select the model and the parameters
problem_type = "regression"
image_model = IGTD(problem= problem_type, scale=[image_size,image_size], fea_dist_method='Euclidean', image_dist_method='Euclidean', error='abs', max_step=30000, val_step=300, random_seed=SEED)
name = f"IGTD_{image_size}x{image_size}_fEuclidean_iEuclidean_abs"

#Define the dataset path and the folder where the images will be saved
images_folder = f"./HyNNImages/Regression/{dataset_name}/images_{dataset_name}_{name}"

In [ ]:
train_loader, val_loader, test_loader, attributes, imgs_shape  = load_and_preprocess_data_test(images_folder, image_model, problem_type, batch_size=batch_size)

### EXPERIMENT 3: REFINED

In [83]:
#Select the model and the parameters
problem_type = "regression"
image_model = REFINED(problem= problem_type, random_seed=SEED, n_processors=64, verbose=True)
name = f"REFINED"

#Define the dataset path and the folder where the images will be saved
images_folder = f"./HyNNImages/Regression/{dataset_name}/images_{dataset_name}_{name}"

In [84]:
train_loader, val_loader, test_loader, attributes, imgs_shape  = load_and_preprocess_data_test(images_folder, image_model, problem_type, batch_size=batch_size)

In [85]:
patch_size = 19
patch_size = 4
patch_size = 8
#patch_size = 20

In [86]:
model_name_f = "ViT"

In [87]:
model_name = f"{name}_Model4_patch{patch_size}_epochs200_batch128"

In [88]:
model_path = f"./models/Regression/{dataset_name}/ViT/{model_name}/best_model.pth"

In [89]:
save_path = f'./logs/Regression/{dataset_name}/ViT/{model_name}'

In [90]:
device="cuda"

In [91]:
model = try_create_model(Model4, patch_size, imgs_shape)

In [92]:
# Make sure you define your model architecture before loading
model = try_create_model(Model4, patch_size, imgs_shape) # <-- replace with your ViT+MLP or hybrid model
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

In [93]:
test_loader

In [94]:
all_x_imgs = []
all_attn_maps_per_layer = None  # Will initialize after seeing first batch
all_real_positions = []
all_pred_positions = []

model.eval()
with torch.no_grad():
    for x_img_batch, y_batch in test_loader:
        x_img_batch = x_img_batch.to(device)
        y_batch = y_batch.to(device)

        preds, attn_maps = model(x_img_batch, return_attn=True)  # attn_maps: List[Tensor] per layer

        # Initialize attention layer storage
        if all_attn_maps_per_layer is None:
            all_attn_maps_per_layer = [[] for _ in range(len(attn_maps))]

        for i, attn in enumerate(attn_maps):
            all_attn_maps_per_layer[i].append(attn.cpu())

        # Store results
        all_x_imgs.append(x_img_batch.cpu())
        all_real_positions.append(y_batch.cpu())
        all_pred_positions.append(preds.cpu())

# Concatenate attention maps per layer
attn_maps = [torch.cat(layer_maps, dim=0) for layer_maps in all_attn_maps_per_layer]  # List of [N, heads, tokens, tokens]

# Concatenate other data
x_imgs = torch.cat(all_x_imgs, dim=0)                 # [N, C, H, W]
real_positions = torch.cat(all_real_positions, dim=0) # [N, 2]
predicted_positions = torch.cat(all_pred_positions, dim=0)  # [N, 2]


In [95]:
save_path

In [96]:
sample_index = 20

In [97]:
y_batch[sample_index]

In [98]:
import joblib

In [99]:
transform_model_path = f"HyNNImages/Regression/{dataset_name}/images_{dataset_name}_{name}/model.pkl"

# Load saved state
state_dict = joblib.load(transform_model_path)

# Load the model
transform_model = REFINED()
transform_model.__dict__.update(state_dict)


In [100]:
def get_feature_coordinates(map_in_int_MDS, feature_names, with_names=False):
    """
    Returns a dictionary mapping each original feature index to its (row, column) position
    in the final image layout after REFINED optimization. Optionally includes feature names.

    Parameters
    ----------
    map_in_int_MDS : np.ndarray
        A 2D array where each cell contains a feature index or -1 (for unused positions).

    feature_names : list or pandas.Index
        List of feature names, ordered by their original index.

    with_names : bool, optional
        If True, includes feature names in the dictionary values.

    Returns
    -------
    feature_to_position : dict
        If with_names=False:
            {feature_idx: (row, col)}
        If with_names=True:
            {feature_idx: {'name': str, 'position': (row, col)}}
    """
    feature_to_position = {}

    for row in range(map_in_int_MDS.shape[0]):
        for col in range(map_in_int_MDS.shape[1]):
            feat_idx = map_in_int_MDS[row, col]
            if feat_idx != -1:
                if with_names:
                    feature_to_position[feat_idx] = {
                        "name": feature_names[feat_idx],
                        "position": (row, col)
                    }
                else:
                    feature_to_position[feat_idx] = (row, col)

    return feature_to_position


In [101]:
feature_coords = get_feature_coordinates(
    transform_model.map_in_int_MDS,
    df.columns,
    with_names=True
)

In [102]:
import matplotlib.pyplot as plt

def show_image(x_img):
    # Convert from CHW → HWC
    img = x_img.permute(1, 2, 0).cpu().numpy().astype(np.uint8)

    # Convert BGR → RGB (OpenCV compatibility)
    img = img[..., ::-1]

    # Plot
    plt.figure(figsize=(4, 4))
    plt.imshow(img, interpolation='nearest')
    plt.title("Original Image")
    plt.axis("off")
    plt.show()


In [103]:
# Usage for sample 0 in batch
show_image(x_imgs[sample_index])

In [104]:
import re

def sort_antenna_names(antenna_list):
    def extract_number(name):
        match = re.search(r'Antenna(\d+)', name)
        return int(match.group(1)) if match else float('inf')
    return sorted(antenna_list, key=extract_number)

def show_angle_module_antenna_masks(x_img, feature_coords, save_path=None):
    """
    Shows separate antenna masks for Angle and Module features,
    using original antenna names directly (no remapping).
    """
    import os
    import re
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

    # Step 1: Convert image CHW → HWC, BGR → RGB
    img = x_img.permute(1, 2, 0).cpu().numpy().astype(np.uint8)[..., ::-1]
    H, W = img.shape[:2]

    # Step 2: Create masks
    mask_angle = np.zeros((H, W), dtype=np.uint8)
    mask_module = np.zeros((H, W), dtype=np.uint8)

    # Step 3: Extract original antenna names
    all_antennas = [v["name"].split("Subcarrier")[0] for v in feature_coords.values()]
    original_antennas = sorted(set(all_antennas), key=lambda x: int(re.search(r'\d+', x).group()))

    antenna_to_id = {ant: i + 1 for i, ant in enumerate(original_antennas)}
    id_to_antenna = {i + 1: ant for i, ant in enumerate(original_antennas)}

    # Step 4: Fill masks
    for v in feature_coords.values():
        row, col = v["position"]
        antenna = v["name"].split("Subcarrier")[0]
        antenna_id = antenna_to_id[antenna]
        if "Angle" in v["name"]:
            mask_angle[row, col] = antenna_id
        elif "Module" in v["name"]:
            mask_module[row, col] = antenna_id

    # Step 5: Plot
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    shared_cmap = ListedColormap(['white'] + list(plt.cm.tab20.colors[:len(original_antennas)]))

    axes[0].imshow(img)
    axes[0].set_title("Original Image", fontsize=44, pad=14)
    axes[0].axis("off")

    axes[1].imshow(mask_angle, cmap=shared_cmap, interpolation="nearest")
    axes[1].set_title("Antenna (Angle)", fontsize=44, pad=14)
    axes[1].axis("off")

    axes[2].imshow(mask_module, cmap=shared_cmap, interpolation="nearest")
    axes[2].set_title("Antenna (Module)", fontsize=44, pad=14)
    axes[2].axis("off")

    # Step 6: Unified legend
    fig.subplots_adjust(bottom=0.2, wspace=0.1)

    handles = [plt.Rectangle((0, 0), 1, 1, color=shared_cmap(i)) for i in range(1, len(original_antennas) + 1)]
    labels = [id_to_antenna[i] for i in range(1, len(original_antennas) + 1)]

    fig.legend(handles, labels, loc='lower center',
               bbox_to_anchor=(0.51, -0.20), fontsize=32,
               title="Antennas", title_fontsize=37, ncol=4)

    # ✅ Save to file
    if save_path:
        os.makedirs(save_path, exist_ok=True)
        full_path = f"{save_path}/{dataset_name}_{model_name_f}_antenna_angle_module_mask.pdf"
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"✅ Antenna mask saved to: {full_path}")

    plt.show()


In [105]:
show_angle_module_antenna_masks(x_imgs[sample_index], feature_coords, save_path)

In [106]:
def show_angle_module_antenna_masks_no_legend(x_img, feature_coords, save_path=None):
    """
    Shows separate antenna masks for Angle and Module features,
    using the same color map, without legends.
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

    # Step 1: Convert image CHW → HWC, BGR → RGB
    img = x_img.permute(1, 2, 0).cpu().numpy().astype(np.uint8)[..., ::-1]
    H, W = img.shape[:2]

    # Step 2: Create masks
    mask_angle = np.zeros((H, W), dtype=np.uint8)
    mask_module = np.zeros((H, W), dtype=np.uint8)

    # Step 3: Use shared antenna names and ID mapping
    all_antennas = [v["name"].split("Subcarrier")[0] for v in feature_coords.values()]
    unique_antennas = sort_antenna_names(set(all_antennas))

    antenna_to_id = {name: i + 1 for i, name in enumerate(unique_antennas)}

    # Step 4: Fill masks
    for v in feature_coords.values():
        row, col = v["position"]
        antenna = v["name"].split("Subcarrier")[0]
        antenna_id = antenna_to_id[antenna]
        if "Angle" in v["name"]:
            mask_angle[row, col] = antenna_id
        elif "Module" in v["name"]:
            mask_module[row, col] = antenna_id

    # Step 5: Plot
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    shared_cmap = ListedColormap(['white'] + list(plt.cm.tab20.colors[:len(unique_antennas)]))

    axes[0].imshow(img)
    axes[0].set_title("Original Image", fontsize=44, pad=14)
    axes[0].axis("off")

    axes[1].imshow(mask_angle, cmap=shared_cmap, interpolation="nearest")
    axes[1].set_title("Antenna (Angle)", fontsize=44, pad=14)
    axes[1].axis("off")

    axes[2].imshow(mask_module, cmap=shared_cmap, interpolation="nearest")
    axes[2].set_title("Antenna (Module)", fontsize=44, pad=14)
    axes[2].axis("off")

    fig.subplots_adjust(wspace=0.2, bottom=0.1)

    # ✅ Save to file
    if save_path:
        os.makedirs(save_path, exist_ok=True)
        full_path = f"{save_path}/{dataset_name}_{model_name_f}_antenna_angle_module_mask_no_legend.pdf"
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"✅ Antenna mask (no legend) saved to: {full_path}")

    plt.show()


In [107]:
show_angle_module_antenna_masks_no_legend(x_imgs[sample_index], feature_coords, save_path)

In [108]:
def compare_attention_layers_avg_by_row(
    x_imgs,
    attn_maps,
    feature_coords,
    patch_grid=None,
    head=0,
    alpha=0.6,
    cmap='viridis',
    save_path=None
):
    import matplotlib.pyplot as plt
    import numpy as np
    import math
    import re
    import os
    from matplotlib.colors import ListedColormap

    img = x_imgs[0].permute(1, 2, 0).cpu().numpy().astype(np.uint8)[..., ::-1]
    H, W = img.shape[:2]
    num_layers = len(attn_maps)

    antenna_names = [v["name"].split("Subcarrier")[0] for v in feature_coords.values()]
    unique_antennas = sorted(set(antenna_names), key=lambda x: int(re.search(r'\d+', x).group()))
    antenna_to_id = {name: i + 1 for i, name in enumerate(unique_antennas)}

    mask = np.zeros((H, W), dtype=np.uint8)
    for v in feature_coords.values():
        row, col = v["position"]
        antenna = v["name"].split("Subcarrier")[0]
        mask[row, col] = antenna_to_id[antenna]

    has_background = (mask == 0).any()
    antenna_cmap = ListedColormap(
        ['white'] + list(plt.cm.tab20.colors[:len(unique_antennas)])
        if has_background else list(plt.cm.tab20.colors[:len(unique_antennas)])
    )
    color_offset = 1 if has_background else 0

    row_figs, row_axes = {}, {}
    for row_type in ['image', 'antenna', 'overlay']:
        if row_type == 'overlay':
            num_cols = math.ceil(num_layers / 2)
            fig, axes = plt.subplots(2, num_cols, figsize=(num_cols * 6, 12))
            axes = axes.flatten()
        else:
            fig, axes = plt.subplots(1, num_layers, figsize=(num_layers * 6, 8))
        if num_layers == 1:
            axes = [axes]
        row_figs[row_type] = fig
        row_axes[row_type] = axes

    for i, layer_attn in enumerate(attn_maps):
        if isinstance(head, int):
            attn_matrix = layer_attn[:, head, 0, 1:]
        elif isinstance(head, str) and head == "mean":
            attn_matrix = layer_attn[:, :, 0, 1:].mean(dim=1)
        else:
            raise ValueError("head must be int or 'mean'")

        attn_vector = attn_matrix.mean(dim=0).detach().cpu().numpy()
        num_patches = attn_vector.shape[0]
        grid_shape = patch_grid or (int(math.sqrt(num_patches)), int(math.sqrt(num_patches)))
        assert np.prod(grid_shape) == num_patches

        attn_map = attn_vector.reshape(grid_shape)
        attn_map_norm = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)

        ax = row_axes["image"][i]
        ax.imshow(img, extent=(0, W, H, 0))
        im0 = ax.imshow(attn_map_norm, cmap=cmap, alpha=alpha, extent=(0, W, H, 0))
        ax.set_title(f"Layer {i+1} (Image)", fontsize=20, pad=7)
        ax.axis("off")
        row_figs["image"].colorbar(im0, ax=ax, fraction=0.046, pad=0.04).set_label("Attention", fontsize=20)

        ax = row_axes["antenna"][i]
        ax.imshow(mask, cmap=antenna_cmap, interpolation='nearest', extent=(0, W, H, 0))
        ax.set_title(f"Layer {i+1} (Antenna)", fontsize=20, pad=7)
        ax.axis("off")

        ax = row_axes["overlay"][i]
        ax.imshow(mask, cmap=antenna_cmap, interpolation='nearest', extent=(0, W, H, 0))
        im2 = ax.imshow(attn_map_norm, cmap=cmap, alpha=alpha, extent=(0, W, H, 0))
        ax.set_title(f"Layer {i+1} (Avg Attention)", fontsize=25, pad=10)
        ax.axis("off")
        cbar = row_figs["overlay"].colorbar(im2, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=19)

    # Legend
    handles, labels = [], []
    if has_background:
        handles.append(plt.Rectangle((0, 0), 1, 1, color='white'))
        labels.append('Not assigned')
    for i, name in enumerate(unique_antennas):
        handles.append(plt.Rectangle((0, 0), 1, 1, color=antenna_cmap(i + color_offset)))
        labels.append(name)

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        for row_type, fig in row_figs.items():
            filename = f"{dataset_name}_{model_name_f}_row_{row_type}.pdf"
            fig.subplots_adjust(bottom=0.2, wspace=0.2)
            fig.savefig(os.path.join(save_path, filename), dpi=300, bbox_inches='tight')
            print(f"✅ Saved: {filename}")
            fig.show()

        fig_legend = plt.figure(figsize=(8, 1.5))
        legend_ax = fig_legend.add_subplot(111)
        legend_ax.axis('off')
        legend_ax.legend(
            handles, labels, loc='center',
            ncol=4, fontsize=20, title="Antennas", title_fontsize=25,
            frameon=True, handletextpad=0.5, columnspacing=1.5
        )
        fig_legend.tight_layout()
        legend_path = os.path.join(save_path, f"{dataset_name}_{model_name_f}_legend.pdf")
        fig_legend.savefig(legend_path, dpi=300, bbox_inches='tight', pad_inches=0.05)
        print(f"✅ Saved: {os.path.basename(legend_path)}")


In [109]:
compare_attention_layers_avg_by_row(
    x_imgs=x_imgs,
    attn_maps=attn_maps,
    feature_coords=feature_coords,
    head='mean',
    save_path=save_path
)

In [110]:
def compare_attention_rollout(
    x_img,
    attn_maps,
    feature_coords,
    patch_grid=None,
    head='mean',
    sample_index=0,
    alpha=0.6,
    cmap='viridis',
    save_path=None
):
    """
    Visualizes attention rollout across all layers (class token to patch tokens)
    with 3 side-by-side plots:
    1. Original Image + Attention
    2. Antenna Layout
    3. Antenna Layout + Attention Overlay
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import torch
    import math
    import re
    from matplotlib.colors import ListedColormap

    img = x_img.permute(1, 2, 0).cpu().numpy().astype(np.uint8)[..., ::-1]
    H, W = img.shape[:2]
    num_tokens = attn_maps[0].shape[-1]
    device = attn_maps[0].device
    rollout = torch.eye(num_tokens, device=device)

    for attn in attn_maps:
        attn_layer = attn[sample_index]
        A = attn_layer.mean(dim=0) if head == 'mean' else attn_layer[head]
        A = A + torch.eye(A.shape[0], device=device)
        A = A / A.sum(dim=-1, keepdim=True)
        rollout = rollout @ A

    attn_vector = rollout[0, 1:].detach().cpu().numpy()

    num_patches = attn_vector.shape[0]
    if patch_grid is None:
        side = int(math.sqrt(num_patches))
        assert side * side == num_patches, "Cannot infer square patch grid"
        grid_shape = (side, side)
    else:
        grid_shape = patch_grid
        assert np.prod(grid_shape) == num_patches

    attn_map = attn_vector.reshape(grid_shape)
    attn_map_norm = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)

    # Build antenna mask
    antenna_names = [v["name"].split("Subcarrier")[0] for v in feature_coords.values()]
    unique_antennas = sorted(set(antenna_names), key=lambda x: int(re.search(r'\d+', x).group()))
    antenna_to_id = {name: i + 1 for i, name in enumerate(unique_antennas)}
    id_to_antenna = {i + 1: name for i, name in enumerate(unique_antennas)}

    mask = np.zeros((H, W), dtype=np.uint8)
    for v in feature_coords.values():
        row, col = v["position"]
        antenna = v["name"].split("Subcarrier")[0]
        mask[row, col] = antenna_to_id[antenna]

    # Use separate colormap for antenna layout
    has_background = (mask == 0).any()
    if has_background:
        antenna_cmap = ListedColormap(['white'] + list(plt.cm.tab20.colors[:len(unique_antennas)]))
        color_offset = 1
    else:
        antenna_cmap = ListedColormap(list(plt.cm.tab20.colors[:len(unique_antennas)]))
        color_offset = 0

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # 1. Original + Attention
    axes[0].imshow(img, extent=(0, W, H, 0))
    im1 = axes[0].imshow(attn_map_norm, cmap=cmap, alpha=alpha, extent=(0, W, H, 0))
    axes[0].set_title("Rollout Attention (Image)")
    axes[0].axis("off")
    fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04).set_label("Attention", fontsize=8)

    # 2. Antenna layout
    axes[1].imshow(mask, cmap=antenna_cmap, interpolation='nearest', extent=(0, W, H, 0))
    axes[1].set_title("Antenna Layout")
    axes[1].axis("off")

    # 3. Antenna + Attention overlay
    axes[2].imshow(mask, cmap=antenna_cmap, interpolation='nearest', extent=(0, W, H, 0))
    im3 = axes[2].imshow(attn_map_norm, cmap=cmap, alpha=alpha, extent=(0, W, H, 0))
    axes[2].set_title("Rollout Attention + Antennas")
    axes[2].axis("off")
    fig.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04).set_label("Attention", fontsize=8)

    # Legend
    handles = []
    labels = []
    if has_background:
        handles.append(plt.Rectangle((0, 0), 1, 1, color='white'))
        labels.append('Not assigned')
    for i in range(len(unique_antennas)):
        handles.append(plt.Rectangle((0, 0), 1, 1, color=antenna_cmap(i + color_offset)))
        labels.append(id_to_antenna[i + 1])

    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05),
               ncol=min(4, len(labels)), fontsize=8)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.0)

    # Save image if path is given
    if save_path:
        os.makedirs(save_path, exist_ok=True)
        full_path = f"{save_path}/{dataset_name}_{model_name_f}_rollout_attention_sample_{sample_index}.pdf"
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"✅ Saved rollout attention plot at: {full_path}")
        
    plt.show()



In [111]:
compare_attention_rollout(
    x_img=x_imgs[sample_index],
    attn_maps=attn_maps,
    feature_coords=feature_coords,
    head='mean',
    sample_index=sample_index
)

In [112]:
def compare_attention_last_layer(
    x_img,
    attn_maps,
    feature_coords,
    patch_grid=None,
    head=0,
    sample_index=0,
    alpha=0.6,
    cmap='viridis',
    save_path=None
):
    """
    Visualizes attention from the last ViT layer:
    - Image + Attention
    - Antenna Layout
    - Antenna + Attention Overlay
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import math
    import re
    from matplotlib.colors import ListedColormap

    img = x_img.permute(1, 2, 0).cpu().numpy().astype(np.uint8)[..., ::-1]
    H, W = img.shape[:2]

    # Step 1: Extract attention vector
    attn_layer = attn_maps[-1][sample_index]
    if isinstance(head, int):
        attn_vector = attn_layer[head, 0, 1:].detach().cpu().numpy()
    elif isinstance(head, str) and head == "mean":
        attn_vector = attn_layer[:, 0, 1:].mean(dim=0).detach().cpu().numpy()
    else:
        raise ValueError("head must be int or 'mean'")

    # Step 2: Infer patch grid and normalize
    num_patches = attn_vector.shape[0]
    if patch_grid is None:
        side = int(math.sqrt(num_patches))
        assert side * side == num_patches
        grid_shape = (side, side)
    else:
        grid_shape = patch_grid
        assert np.prod(grid_shape) == num_patches

    attn_map = attn_vector.reshape(grid_shape)
    attn_map_norm = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)

    # Step 3: Build antenna mask
    antenna_names = [v["name"].split("Subcarrier")[0] for v in feature_coords.values()]
    unique_antennas = sorted(set(antenna_names), key=lambda x: int(re.search(r'\d+', x).group()))
    antenna_to_id = {name: i + 1 for i, name in enumerate(unique_antennas)}
    id_to_antenna = {i + 1: name for i, name in enumerate(unique_antennas)}

    mask = np.zeros((H, W), dtype=np.uint8)
    for v in feature_coords.values():
        row, col = v["position"]
        antenna = v["name"].split("Subcarrier")[0]
        mask[row, col] = antenna_to_id[antenna]

    # Step 4: Separate antenna colormap
    has_background = (mask == 0).any()
    if has_background:
        antenna_cmap = ListedColormap(['white'] + list(plt.cm.tab20.colors[:len(unique_antennas)]))
        color_offset = 1
    else:
        antenna_cmap = ListedColormap(list(plt.cm.tab20.colors[:len(unique_antennas)]))
        color_offset = 0

    # Step 5: Plot panels
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Panel 1: Image + attention
    axes[0].imshow(img, extent=(0, W, H, 0))
    im1 = axes[0].imshow(attn_map_norm, cmap=cmap, alpha=alpha, extent=(0, W, H, 0))
    axes[0].set_title("Last Layer Attention (Image)")
    axes[0].axis("off")
    fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04).set_label("Attention", fontsize=8)

    # Panel 2: Antenna layout
    axes[1].imshow(mask, cmap=antenna_cmap, interpolation='nearest', extent=(0, W, H, 0))
    axes[1].set_title("Antenna Layout")
    axes[1].axis("off")

    # Panel 3: Antenna + attention overlay
    axes[2].imshow(mask, cmap=antenna_cmap, interpolation='nearest', extent=(0, W, H, 0))
    im3 = axes[2].imshow(attn_map_norm, cmap=cmap, alpha=alpha, extent=(0, W, H, 0))
    axes[2].set_title("Attention + Antennas")
    axes[2].axis("off")
    fig.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04).set_label("Attention", fontsize=8)

    # Step 6: Add legend
    handles, labels = [], []
    if has_background:
        handles.append(plt.Rectangle((0, 0), 1, 1, color='white'))
        labels.append('Not assigned')
    for i in range(len(unique_antennas)):
        handles.append(plt.Rectangle((0, 0), 1, 1, color=antenna_cmap(i + color_offset)))
        labels.append(id_to_antenna[i + 1])

    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05),
               ncol=min(4, len(labels)), fontsize=8)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.0)

    # Save if requested
    if save_path:
        os.makedirs(save_path, exist_ok=True)
        full_path = f"{save_path}/{dataset_name}_{model_name_f}_last_layer_attention_sample_{sample_index}.pdf"
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"✅ Saved last layer attention to: {full_path}")
        
    plt.show()



In [113]:
compare_attention_last_layer(
    x_img=x_imgs[sample_index],
    attn_maps=attn_maps,
    feature_coords=feature_coords,
    head='mean',
    sample_index=sample_index,
    save_path=save_path
)

In [114]:
import matplotlib.pyplot as plt
import numpy as np
import math
import re
from matplotlib.colors import ListedColormap

def compare_attention_last_layer_point(
    x_img,
    attn_maps,
    feature_coords,
    real_position=None,        # tensor([x, y])
    predicted_position=None,   # tensor([x, y])
    patch_grid=None,
    head=0,
    sample_index=0,
    alpha=0.6,
    cmap='viridis',
    save_path=None

):
    """
    Two-row visualization:
    [Image + Attention, Antenna Layout]
    [Attention + Antennas, Real + Predicted Position View with Split Lines]
    """

    img = x_img.permute(1, 2, 0).cpu().numpy().astype(np.uint8)[..., ::-1]
    H, W = img.shape[:2]

    # --- Extract attention ---
    attn_layer = attn_maps[-1][sample_index]
    if isinstance(head, int):
        attn_vector = attn_layer[head, 0, 1:].detach().cpu().numpy()
    elif isinstance(head, str) and head == "mean":
        attn_vector = attn_layer[:, 0, 1:].mean(dim=0).detach().cpu().numpy()
    else:
        raise ValueError("head must be int or 'mean'")

    # --- Patch grid ---
    num_patches = attn_vector.shape[0]
    if patch_grid is None:
        side = int(math.sqrt(num_patches))
        assert side * side == num_patches
        grid_shape = (side, side)
    else:
        grid_shape = patch_grid
        assert np.prod(grid_shape) == num_patches

    attn_map = attn_vector.reshape(grid_shape)
    attn_map_norm = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)

    # --- Antenna mask ---
    antenna_names = [v["name"].split("Subcarrier")[0] for v in feature_coords.values()]
    unique_antennas = sorted(set(antenna_names), key=lambda x: int(re.search(r'\d+', x).group()))
    antenna_to_id = {name: i + 1 for i, name in enumerate(unique_antennas)}
    id_to_antenna = {i + 1: name for i, name in enumerate(unique_antennas)}

    mask = np.zeros((H, W), dtype=np.uint8)
    for v in feature_coords.values():
        row, col = v["position"]
        antenna = v["name"].split("Subcarrier")[0]
        mask[row, col] = antenna_to_id[antenna]

    # --- Antenna colormap ---
    has_background = (mask == 0).any()
    if has_background:
        antenna_cmap = ListedColormap(['white'] + list(plt.cm.tab20.colors[:len(unique_antennas)]))
        color_offset = 1
    else:
        antenna_cmap = ListedColormap(list(plt.cm.tab20.colors[:len(unique_antennas)]))
        color_offset = 0

    # --- Plot 2×2 layout ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # Top-left: Image + Attention
    axes[0, 0].imshow(img, extent=(0, W, H, 0))
    im0 = axes[0, 0].imshow(attn_map_norm, cmap=cmap, alpha=alpha, extent=(0, W, H, 0))
    axes[0, 0].set_title("Image + Attention")
    axes[0, 0].axis("off")
    fig.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04).set_label("Attention", fontsize=8)

    # Top-right: Antenna Layout
    axes[0, 1].imshow(mask, cmap=antenna_cmap, interpolation='nearest', extent=(0, W, H, 0))
    axes[0, 1].set_title("Antenna Layout")
    axes[0, 1].axis("off")

    # Bottom-left: Attention + Antennas
    axes[1, 0].imshow(mask, cmap=antenna_cmap, interpolation='nearest', extent=(0, W, H, 0))
    im1 = axes[1, 0].imshow(attn_map_norm, cmap=cmap, alpha=alpha, extent=(0, W, H, 0))
    axes[1, 0].set_title("Attention + Antennas")
    axes[1, 0].axis("off")
    fig.colorbar(im1, ax=axes[1, 0], fraction=0.046, pad=0.04).set_label("Attention", fontsize=8)

    # Bottom-right: Real + Predicted Positions
    axes[1, 1].set_title("Real & Predicted Position")
    axes[1, 1].set_xlabel("Position X")
    axes[1, 1].set_ylabel("Position Y")
    axes[1, 1].grid(True)

    # Set bounds from known dataset
    min_x, max_x = -1437, 1357
    min_y, max_y = 1155, 4029
    padding_x = 0.05 * (max_x - min_x)
    padding_y = 0.05 * (max_y - min_y)

    axes[1, 1].set_xlim(min_x - padding_x, max_x + padding_x)
    axes[1, 1].set_ylim(min_y - padding_y, max_y + padding_y)

    # Add quadrant split
    axes[1, 1].axhline((min_y + max_y) / 2, color='black', linestyle='--', linewidth=1)
    axes[1, 1].axvline((min_x + max_x) / 2, color='black', linestyle='--', linewidth=1)

    # Keep square layout
    axes[1, 1].set_aspect((max_x - min_x) / (max_y - min_y))

    # Plot real position
    if real_position is not None:
        pos = real_position.detach().cpu().numpy()
        axes[1, 1].scatter(pos[0], pos[1], color='red', s=60, marker='o', label=f"Real ({pos[0]:.0f}, {pos[1]:.0f})")

    # Plot predicted position
    if predicted_position is not None:
        pred = predicted_position.detach().cpu().numpy()
        axes[1, 1].scatter(pred[0], pred[1], color='blue', s=60, marker='x', label=f"Predicted ({pred[0]:.0f}, {pred[1]:.0f})")

    # Position legend inside the real vs pred plot
    axes[1, 1].legend(fontsize=8, loc='upper right')

    # --- Legend (Antenna IDs) ---
    handles, labels = [], []
    if has_background:
        handles.append(plt.Rectangle((0, 0), 1, 1, color='white'))
        labels.append('Not assigned')
    for i in range(len(unique_antennas)):
        handles.append(plt.Rectangle((0, 0), 1, 1, color=antenna_cmap(i + color_offset)))
        labels.append(id_to_antenna[i + 1])
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05),
               ncol=min(6, len(labels)), fontsize=8)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.05)

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        full_path = f"{save_path}/{dataset_name}_{model_name_f}_attn_last_layer_point_sample_{sample_index}.pdf"
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"✅ Saved to: {full_path}")
        
    plt.show()


In [115]:
compare_attention_last_layer_point(
    x_img=x_imgs[sample_index],
    attn_maps=attn_maps,
    feature_coords=feature_coords,
    head='mean',
    sample_index=sample_index,
    real_position=real_positions[sample_index],
    predicted_position=predicted_positions[sample_index],
    save_path=save_path
)

In [116]:
import matplotlib.pyplot as plt
import numpy as np
import math
import re
import os
from matplotlib.colors import ListedColormap
from brokenaxes import brokenaxes
from matplotlib.gridspec import GridSpec


def compare_attention_batch_average(
    x_imgs,
    attn_maps,
    feature_coords,
    real_positions=None,
    predicted_positions=None,
    patch_grid=None,
    head=0,
    alpha=0.6,
    cmap='viridis',
    save_path=None
):
    img = x_imgs[0].permute(1, 2, 0).cpu().numpy().astype(np.uint8)[..., ::-1]
    H, W = img.shape[:2]

    attn_layer = attn_maps[-1]
    if isinstance(head, int):
        attn_matrix = attn_layer[:, head, 0, 1:]
    elif isinstance(head, str) and head == "mean":
        attn_matrix = attn_layer[:, :, 0, 1:].mean(dim=1)
    else:
        raise ValueError("head must be int or 'mean'")

    attn_vector = attn_matrix.mean(dim=0).detach().cpu().numpy()

    num_patches = attn_vector.shape[0]
    if patch_grid is None:
        side = int(math.sqrt(num_patches))
        assert side * side == num_patches
        grid_shape = (side, side)
    else:
        grid_shape = patch_grid
        assert np.prod(grid_shape) == num_patches

    attn_map = attn_vector.reshape(grid_shape)
    attn_map_norm = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)

    antenna_names = [v["name"].replace("Antenna", "Ant").split("Subcarrier")[0] for v in feature_coords.values()]
    unique_antennas = sorted(set(antenna_names), key=lambda x: int(re.search(r'\d+', x).group()))
    antenna_to_id = {name: i + 1 for i, name in enumerate(unique_antennas)}

    mask = np.zeros((H, W), dtype=np.uint8)
    for v in feature_coords.values():
        row, col = v["position"]
        antenna = v["name"].replace("Antenna", "Ant").split("Subcarrier")[0]
        mask[row, col] = antenna_to_id[antenna]

    has_background = (mask == 0).any()
    antenna_cmap = ListedColormap(['white'] + list(plt.cm.tab20.colors[:len(unique_antennas)])) if has_background else ListedColormap(list(plt.cm.tab20.colors[:len(unique_antennas)]))
    color_offset = 1 if has_background else 0

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    axes[0, 0].imshow(img, extent=(0, W, H, 0))
    im0 = axes[0, 0].imshow(attn_map_norm, cmap=cmap, alpha=alpha, extent=(0, W, H, 0))
    axes[0, 0].set_title("Average Attention (All Samples)")
    axes[0, 0].axis("off")
    fig.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04).set_label("Attention", fontsize=8)

    axes[0, 1].imshow(mask, cmap=antenna_cmap, interpolation='nearest', extent=(0, W, H, 0))
    axes[0, 1].set_title("Antenna Layout")
    axes[0, 1].axis("off")

    axes[1, 0].imshow(mask, cmap=antenna_cmap, interpolation='nearest', extent=(0, W, H, 0))
    im1 = axes[1, 0].imshow(attn_map_norm, cmap=cmap, alpha=alpha, extent=(0, W, H, 0))
    axes[1, 0].set_title("Avg Attention + Antennas")
    axes[1, 0].axis("off")
    fig.colorbar(im1, ax=axes[1, 0], fraction=0.046, pad=0.04).set_label("Attention", fontsize=8)

    min_x, max_x = -1437, 1357
    min_y, max_y = 1155, 4029
    padding_x = 0.05 * (max_x - min_x)
    padding_y = 0.05 * (max_y - min_y)

    axes[1, 1].set_title("All Real & Predicted Positions")
    axes[1, 1].set_xlabel("Position X")
    axes[1, 1].set_ylabel("Position Y")
    axes[1, 1].grid(True)
    axes[1, 1].set_xlim(min_x - padding_x, max_x + padding_x)
    axes[1, 1].set_ylim(min_y - padding_y, max_y + padding_y)
    axes[1, 1].axhline((min_y + max_y) / 2, color='black', linestyle='--', linewidth=1)
    axes[1, 1].axvline((min_x + max_x) / 2, color='black', linestyle='--', linewidth=1)
    axes[1, 1].set_aspect((max_x - min_x) / (max_y - min_y))

    if real_positions is not None:
        real_np = real_positions.detach().cpu().numpy()
        axes[1, 1].scatter(real_np[:, 0], real_np[:, 1], color='red', s=10, alpha=0.6, label="Real")

    if predicted_positions is not None:
        pred_np = predicted_positions.detach().cpu().numpy()
        axes[1, 1].scatter(pred_np[:, 0], pred_np[:, 1], color='blue', s=10, alpha=0.6, label="Predicted")

    axes[1, 1].legend(fontsize=8, loc='upper right')

    handles, labels = [], []
    if has_background:
        handles.append(plt.Rectangle((0, 0), 1, 1, color='white'))
        labels.append('Not assigned')
    for i, name in enumerate(unique_antennas):
        handles.append(plt.Rectangle((0, 0), 1, 1, color=antenna_cmap(i + color_offset)))
        labels.append(name)

    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05),
               ncol=min(6, len(labels)), fontsize=8)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.1)

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        full_path = f"{save_path}/{dataset_name}_{model_name_f}_average_attention_point.pdf"
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"✅ Saved to: {full_path}")

    plt.show()

    if real_positions is None:
        return

    B = attn_matrix.shape[0]
    h, w = grid_shape
    patch_height = H // h
    patch_width = W // w

    mid_x = (min_x + max_x) / 2
    mid_y = (min_y + max_y) / 2
    quadrant_names = ['Top Left', 'Top Right', 'Bottom Left', 'Bottom Right']
    quadrant_scores = {q: {} for q in quadrant_names}

    for i in range(B):
        x, y = real_positions[i].detach().cpu().numpy()
        if x < mid_x and y < mid_y:
            quadrant = 'Top Left'
        elif x >= mid_x and y < mid_y:
            quadrant = 'Top Right'
        elif x < mid_x and y >= mid_y:
            quadrant = 'Bottom Left'
        else:
            quadrant = 'Bottom Right'

        attn_vec = attn_matrix[i].detach().cpu().numpy().reshape((h, w))

        antenna_scores = {}
        for v in feature_coords.values():
            row, col = v["position"]
            patch_row = min(row * h // H, h - 1)
            patch_col = min(col * w // W, w - 1)
            patch_attn = attn_vec[patch_row, patch_col]
            antenna = v["name"].replace("Antenna", "Ant").split("Subcarrier")[0]
            if antenna not in antenna_scores:
                antenna_scores[antenna] = []
            antenna_scores[antenna].append(patch_attn)

        for ant, vals in antenna_scores.items():
            if ant not in quadrant_scores[quadrant]:
                quadrant_scores[quadrant][ant] = []
            quadrant_scores[quadrant][ant].extend(vals)

    quadrant_avg = {}
    for q in quadrant_names:
        avg_scores = {a: np.mean(v) for a, v in quadrant_scores[q].items()}
        total = sum(avg_scores.values()) + 1e-8
        norm = {a: v / total for a, v in avg_scores.items()}
        sorted_items = sorted(norm.items(), key=lambda x: x[1], reverse=True)
        quadrant_avg[q] = sorted_items

    fig_bar = plt.figure(figsize=(13, 16))
    gs = GridSpec(2, 2, figure=fig_bar, hspace=0.2, wspace=0.35)

    for idx, q in enumerate(quadrant_names):
        sorted_items = quadrant_avg[q]
        if not sorted_items:
            continue

        names = [x[0] for x in sorted_items]
        scores = [x[1] for x in sorted_items]
        colors = [antenna_cmap.colors[antenna_to_id[name] + color_offset - 1] for name in names]

        row, col = divmod(idx, 2)
        bax = brokenaxes(
            xlims=((0, 0.4), (0.9, 1.0)),
            hspace=0.05,
            fig=fig_bar,
            subplot_spec=gs[row, col],
            despine=False,
            d = 0.005
        )

        bax.barh(names, scores, color=colors)
        bax.set_title(f"{q} - Top Antennas", fontsize=23, pad=10)
        bax.set_xlabel("Avg Attention", fontsize=21, labelpad=20)
        bax.tick_params(axis='x', labelsize=15)
        bax.tick_params(axis='y', labelsize=18)
        bax.invert_yaxis()
        
    if save_path:
        os.makedirs(save_path, exist_ok=True)
        full_path = os.path.join(save_path, f"{dataset_name}_{model_name_f}_test_set_average_quadrant_bar_chart_point_brokenax.pdf")
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"✅ Saved to: {full_path}")

    plt.show()


In [117]:
compare_attention_batch_average(
    x_imgs=x_imgs,
    attn_maps=attn_maps,
    feature_coords=feature_coords,
    head='mean',
    real_positions=real_positions,
    predicted_positions=predicted_positions,
    save_path=save_path
)

In [118]:
def compare_attention_angle_vs_module(
    x_imgs,
    attn_maps,
    feature_coords,
    patch_grid=None,
    head=0,
    alpha=0.6,
    cmap='viridis',
    save_path=None
):
    import os
    import matplotlib.pyplot as plt
    import numpy as np
    import math

    img = x_imgs[0].permute(1, 2, 0).cpu().numpy().astype(np.uint8)[..., ::-1]
    H, W = img.shape[:2]

    attn_layer = attn_maps[-1]  # [B, heads, tokens, tokens]
    if isinstance(head, int):
        attn_matrix = attn_layer[:, head, 0, 1:]  # [B, tokens]
    elif isinstance(head, str) and head == "mean":
        attn_matrix = attn_layer[:, :, 0, 1:].mean(dim=1)
    else:
        raise ValueError("head must be int or 'mean'")

    B, num_patches = attn_matrix.shape

    if patch_grid is None:
        side = int(math.sqrt(num_patches))
        assert side * side == num_patches
        grid_shape = (side, side)
    else:
        grid_shape = patch_grid
        assert np.prod(grid_shape) == num_patches

    h, w = grid_shape
    patch_height = H // h
    patch_width = W // w

    angle_total = 0.0
    module_total = 0.0

    print("📦 Iterating over samples and patches:\n")
    for i in range(B):
        attn_vec = attn_matrix[i].detach().cpu().numpy().reshape((h, w))

        patch_counts = {}
        for v in feature_coords.values():
            row, col = v["position"]
            patch_row = min(row * h // H, h - 1)
            patch_col = min(col * w // W, w - 1)
            patch = (patch_row, patch_col)
            ftype = "Angle" if "Angle" in v["name"] else "Module" if "Module" in v["name"] else None
            if ftype:
                if patch not in patch_counts:
                    patch_counts[patch] = {"Angle": 0, "Module": 0}
                patch_counts[patch][ftype] += 1

        for (patch_row, patch_col), counts in sorted(patch_counts.items()):
            patch_attn = attn_vec[patch_row, patch_col]
            total = counts["Angle"] + counts["Module"]
            if total == 0:
                continue

            angle_count = counts["Angle"]
            module_count = counts["Module"]

            angle_portion = angle_count / total
            module_portion = module_count / total
            angle_share = patch_attn * angle_portion
            module_share = patch_attn * module_portion

            angle_total += angle_share
            module_total += module_share

    total_attention = angle_total + module_total + 1e-8
    norm_angle = angle_total / total_attention
    norm_module = module_total / total_attention

    print("\n🔎 Final Totals Across All Samples:")
    print(f"Angle Total Attention:  {angle_total:.6f}")
    print(f"Module Total Attention: {module_total:.6f}")
    print(f"Normalized: Angle = {norm_angle:.4f}, Module = {norm_module:.4f}")

    # -- Plot 2: Angle vs Module Bar Chart
    plt.figure(figsize=(4, 5))
    plt.bar(["Angle", "Module"], [norm_angle, norm_module], color=["tab:orange", "tab:blue"])
    plt.ylim(0, 1)
    plt.ylabel("Normalized Attention", fontsize=20)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.title("Attention Distribution:\nAngle vs Module", fontsize=24)

    if save_path:
        bar_path = f"{save_path}/{dataset_name}_{model_name_f}_attention_angle_module_bar.pdf"
        plt.savefig(bar_path, dpi=300, bbox_inches='tight')
        print(f"✅ Saved bar chart to: {bar_path}")

    plt.show()


In [119]:
compare_attention_angle_vs_module(
    x_imgs=x_imgs,
    attn_maps=attn_maps,
    feature_coords=feature_coords,
    head='mean',
    save_path=save_path
)

In [120]:
import matplotlib.pyplot as plt
import numpy as np
import math
import re
import os
from matplotlib.colors import ListedColormap
from brokenaxes import brokenaxes  # Make sure brokenaxes is installed

def compare_attention_average_antennas_only(
    x_imgs,
    attn_maps,
    feature_coords,
    real_positions=None,
    patch_grid=None,
    head=0,
    alpha=0.6,
    cmap='viridis',
    save_path=None
):
    img = x_imgs[0].permute(1, 2, 0).cpu().numpy().astype(np.uint8)[..., ::-1]
    H, W = img.shape[:2]

    attn_layer = attn_maps[-1]
    if isinstance(head, int):
        attn_matrix = attn_layer[:, head, 0, 1:]
    elif isinstance(head, str) and head == "mean":
        attn_matrix = attn_layer[:, :, 0, 1:].mean(dim=1)
    else:
        raise ValueError("head must be int or 'mean'")

    num_patches = attn_matrix.shape[1]
    if patch_grid is None:
        side = int(math.sqrt(num_patches))
        assert side * side == num_patches
        grid_shape = (side, side)
    else:
        grid_shape = patch_grid
        assert np.prod(grid_shape) == num_patches

    h, w = grid_shape
    B = attn_matrix.shape[0]
    
    # Step 1: Use shortened antenna names
    antenna_names = [v["name"].replace("Antenna", "Ant").split("Subcarrier")[0] for v in feature_coords.values()]
    unique_antennas = sorted(set(antenna_names), key=lambda x: int(re.search(r'\d+', x).group()))
    antenna_scores = {ant: [] for ant in unique_antennas}

    # Step 2: Collect attention (same logic, updated naming)
    for i in range(B):
        attn_vec = attn_matrix[i].detach().cpu().numpy().reshape((h, w))
        for v in feature_coords.values():
            row, col = v["position"]
            patch_row = min(row * h // H, h - 1)
            patch_col = min(col * w // W, w - 1)
            patch_attn = attn_vec[patch_row, patch_col]
            antenna = v["name"].replace("Antenna", "Ant").split("Subcarrier")[0]
            antenna_scores[antenna].append(patch_attn)

    # Step 3: Normalize and sort
    avg_scores = {a: np.mean(v) for a, v in antenna_scores.items()}
    total = sum(avg_scores.values()) + 1e-8
    norm_scores = {a: v / total for a, v in avg_scores.items()}
    sorted_items = sorted(norm_scores.items(), key=lambda x: x[1], reverse=True)

    names = [x[0] for x in sorted_items]
    scores = [x[1] for x in sorted_items]
    y_pos = np.arange(len(names))

    # Step 4: Color mapping by original name
    color_map = {ant: plt.cm.tab20(i % 20) for i, ant in enumerate(unique_antennas)}
    bar_colors = [color_map[name] for name in names]

    # Step 5: Plot with brokenaxes
    fig = plt.figure(figsize=(8, 11))
    bax = brokenaxes(xlims=((0, 0.4), (0.9, 1.0)), wspace=0.2, despine=False)
    bax.barh(y_pos, scores, color=bar_colors)
    for ax in bax.axs:
        ax.set_yticks(y_pos)
        ax.set_yticklabels(names)
        
    bax.set_xlabel("Normalized Average Attention", fontsize=32, labelpad=30)
    bax.set_title("Per-Antenna Average Attention", fontsize=35, pad=12)
    bax.tick_params(axis='x', labelsize=22)
    bax.tick_params(axis='y', labelsize=26)
    bax.invert_yaxis()

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        full_path = os.path.join(save_path, f"{dataset_name}_{model_name_f}_test_set_average_antenna_bar_chart_brokenx.pdf")
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"✅ Saved to: {full_path}")

    plt.show()


In [121]:
compare_attention_average_antennas_only(
    x_imgs,
    attn_maps,
    feature_coords,
    real_positions=real_positions,
    head="mean",
    save_path=save_path
)

In [122]:
import matplotlib.pyplot as plt
import numpy as np
import math
import re
import os

def compare_attention_per_antenna_by_type_sorted(
    x_imgs,
    attn_maps,
    feature_coords,
    patch_grid=None,
    head=0,
    alpha=0.6,
    cmap='tab20',
    save_path=None,
):
    img = x_imgs[0].permute(1, 2, 0).cpu().numpy().astype(np.uint8)[..., ::-1]
    H, W = img.shape[:2]

    attn_layer = attn_maps[-1]
    if isinstance(head, int):
        attn_matrix = attn_layer[:, head, 0, 1:]
    elif isinstance(head, str) and head == "mean":
        attn_matrix = attn_layer[:, :, 0, 1:].mean(dim=1)
    else:
        raise ValueError("head must be int or 'mean'")

    B, num_patches = attn_matrix.shape
    if patch_grid is None:
        side = int(math.sqrt(num_patches))
        assert side * side == num_patches
        grid_shape = (side, side)
    else:
        grid_shape = patch_grid
        assert np.prod(grid_shape) == num_patches

    h, w = grid_shape

    # Step 1: Get all antenna base names
    antenna_names = [v["name"].split("Subcarrier")[0] for v in feature_coords.values()]
    original_antennas = sorted(set(antenna_names), key=lambda x: int(re.search(r'\d+', x).group()))

    # Step 2: Collect attention per antenna/type (original names)
    scores = {ant: {"Angle": [], "Module": []} for ant in original_antennas}

    for i in range(B):
        attn_vec = attn_matrix[i].detach().cpu().numpy().reshape((h, w))
        for v in feature_coords.values():
            row, col = v["position"]
            patch_row = min(row * h // H, h - 1)
            patch_col = min(col * w // W, w - 1)
            patch_attn = attn_vec[patch_row, patch_col]

            orig_antenna = v["name"].split("Subcarrier")[0]
            ftype = "Angle" if "Angle" in v["name"] else "Module" if "Module" in v["name"] else None
            if ftype:
                scores[orig_antenna][ftype].append(patch_attn)

    # Step 3: Compute averages per pair (antenna, t
    all_items = []
    for antenna, val in scores.items():
        for ftype in ["Angle", "Module"]:
            if val[ftype]:
                avg_score = np.mean(val[ftype])
                all_items.append((f"{antenna} {ftype}", avg_score, antenna))

    # Step 4: Sort by attention
    all_items_sorted = sorted(all_items, key=lambda x: x[1], reverse=True)

    names = [x[0] for x in all_items_sorted]
    values = [x[1] for x in all_items_sorted]
    antennas = [x[2] for x in all_items_sorted]

    # Step 5: Color mapping by base antenna
    unique_antennas = sorted(set(antennas), key=lambda x: int(re.search(r'\d+', x).group()))
    color_map = {ant: plt.get_cmap(cmap)(i % 20) for i, ant in enumerate(unique_antennas)}
    colors = [color_map[ant] for ant in antennas]

    # Step 6: Plot
    fig, ax = plt.subplots(figsize=(8, 16))
    y_pos = np.arange(len(names))
    ax.barh(y_pos, values, color=colors)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names, fontsize=16)
    ax.set_xlabel("Average Attention", fontsize=25)
    ax.set_title("Average Attention per Feature", fontsize=35, pad=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=27)
    ax.set_xlim(0, 1)
    ax.invert_yaxis()

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        full_path = os.path.join(save_path, f"{dataset_name}_{model_name_f}_antenna_angle_module_sorted.pdf")
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"✅ Saved to: {full_path}")

    plt.show()


In [123]:
compare_attention_per_antenna_by_type_sorted(
    x_imgs,
    attn_maps,
    feature_coords,
    head="mean",
    save_path=save_path
)

In [124]:
def compute_attention_entropy(attn_maps, cls_token_index=0, normalize=True):
    """
    Computes attention entropy per head and per layer from a list of attention maps.
    
    Parameters:
        attn_maps: List[torch.Tensor] — list of [B, H, T, T] tensors (one per layer)
        cls_token_index: int — position of CLS token (usually 0)
        normalize: bool — normalize by log(num_tokens)

    Returns:
        entropy_matrix: np.ndarray of shape [num_layers, num_heads]
    """
    num_layers = len(attn_maps)
    B, H, T, _ = attn_maps[0].shape
    entropy_matrix = np.zeros((num_layers, H))

    for l, attn_layer in enumerate(attn_maps):  # attn_layer: [B, H, T, T]
        for h in range(H):
            head_entropy = []
            for b in range(B):
                attn_vector = attn_layer[b, h, cls_token_index, 1:]  # exclude CLS
                attn_vector = attn_vector / (attn_vector.sum() + 1e-8)
                entropy = -torch.sum(attn_vector * torch.log(attn_vector + 1e-8)).item()
                head_entropy.append(entropy)
            avg_entropy = np.mean(head_entropy)
            if normalize:
                max_entropy = np.log(T - 1)
                avg_entropy /= (max_entropy + 1e-8)
            entropy_matrix[l, h] = avg_entropy

    return entropy_matrix


In [125]:
# How concentrated and spread out the attention is in each layer and head

In [126]:
def plot_entropy_heatmap(entropy_matrix, model_name="ViT", cmap="YlGnBu", save_path=None):
    import matplotlib.pyplot as plt
    import os
    import numpy as np

    plt.figure(figsize=(10, 6))
    im = plt.imshow(entropy_matrix, cmap=cmap, aspect='auto', vmin=0.0, vmax=1.0)

    # Configure colorbar
    cbar = plt.colorbar(im, ticks=np.linspace(0.0, 1.0, 6))  # 6 ticks: 0.0, 0.2, ..., 1.0
    cbar.ax.tick_params(labelsize=25)

    # Axis labels and title
    plt.xlabel("Heads", fontsize=30)
    plt.ylabel("Layers", fontsize=30)
    plt.title(f"Attention Entropy Heatmap", fontsize=35, pad=12)
    plt.xticks(range(entropy_matrix.shape[1]), fontsize=30)
    plt.yticks(range(entropy_matrix.shape[0]), fontsize=30)
    plt.tight_layout()

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        full_path = os.path.join(save_path, f"{dataset_name}_{model_name_f}_attention_entropy_heatmap.pdf")
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"✅ Saved to: {full_path}")

    plt.show()


In [127]:
entropy_matrix = compute_attention_entropy(attn_maps)
plot_entropy_heatmap(entropy_matrix, model_name=model_name_f, save_path=save_path)

In [128]:
import torch
import numpy as np
from scipy.stats import spearmanr, pearsonr, ttest_ind
import matplotlib.pyplot as plt
import os

def analyze_attention_entropy_vs_error(
    attn_maps,
    real_position,
    predicted_position,
    cls_token_index=0,
    task='regression',
    normalize=True,
    save_path=None,             # e.g. 'results/'
    alpha=0.6,
    cmap='viridis',
    sample_index=None
):
    """
    Computes per-sample attention entropy and correlates with prediction error.
    Saves scatter plot using dataset and model name if save_path is provided.
    """
    device = attn_maps[0].device
    num_layers = len(attn_maps)
    B, H, T, _ = attn_maps[0].shape

    real_position = real_position.detach().cpu().numpy() if torch.is_tensor(real_position) else np.array(real_position)
    predicted_position = predicted_position.detach().cpu().numpy() if torch.is_tensor(predicted_position) else np.array(predicted_position)

    entropies = []

    sample_indices = [sample_index] if sample_index is not None else range(B)

    for b in sample_indices:
        sample_entropy = []
        for l in range(num_layers):
            for h in range(H):
                attn_vector = attn_maps[l][b, h, cls_token_index, 1:]
                attn_vector = attn_vector / (attn_vector.sum() + 1e-8)
                entropy = -torch.sum(attn_vector * torch.log(attn_vector + 1e-8)).item()
                if normalize:
                    entropy /= (np.log(T - 1) + 1e-8)
                sample_entropy.append(entropy)
        entropies.append(np.mean(sample_entropy))

    entropies = np.array(entropies)

    if task == 'regression':
        errors = np.linalg.norm(real_position - predicted_position, axis=1)
        y_label = "Prediction Error (L2)"
        title = "Attention Entropy vs. Prediction Error"
    elif task == 'classification':
        preds = np.argmax(predicted_position, axis=1) if predicted_position.ndim > 1 else (predicted_position > 0.5).astype(int)
        errors = (preds != real_position).astype(int)
        y_label = "Misclassification (0/1)"
        title = "Attention Entropy vs. Misclassification"
    else:
        raise ValueError("task must be 'regression' or 'classification'")

    # Correlation
    spearman_corr, sp_pval = spearmanr(entropies, errors)
    pearson_corr, pe_pval = pearsonr(entropies, errors)

    # Optional t-test for classification
    if task == 'classification':
        correct = entropies[errors == 0]
        incorrect = entropies[errors == 1]
        t_stat, t_pval = ttest_ind(correct, incorrect)
    else:
        t_stat = t_pval = None

    # Plot
    if sample_index is None:
        plt.figure(figsize=(6, 4))
        plt.scatter(entropies, errors, alpha=alpha, cmap=cmap)
        plt.xlabel("Average Attention Entropy", fontsize=20, labelpad=10)
        plt.ylabel(y_label, fontsize=20)
        plt.title(f"{title}\nSpearman: {spearman_corr:.2f}, Pearson: {pearson_corr:.2f}", fontsize=24)
        plt.xticks(fontsize=15)
        plt.yticks(fontsize=15)
        plt.grid(False)

        if save_path and dataset_name and model_name_f:
            os.makedirs(save_path, exist_ok=True)
            filename = f"{dataset_name}_{model_name_f}_attention_entropy_vs_error_no_grid.pdf"
            full_path = os.path.join(save_path, filename)
            plt.savefig(full_path, dpi=300, bbox_inches='tight')
            print(f"✅ Saved plot to: {full_path}")
        else:
            plt.tight_layout()
            plt.show()

    return {
        'spearman_corr': spearman_corr,
        'spearman_pval': sp_pval,
        'pearson_corr': pearson_corr,
        'pearson_pval': pe_pval,
        'ttest_stat': t_stat,
        'ttest_pval': t_pval,
        'entropies': entropies,
        'errors': errors
    }


In [ ]:
results = analyze_attention_entropy_vs_error(
    attn_maps=attn_maps,
    real_position=real_positions,            # shape [B, 2] for regression
    predicted_position=predicted_positions,       # same shape
    task='regression',
    save_path=save_path
)
